In [1]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [2]:
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('diabets')

In [3]:
data = load_diabetes(scaled=False)
X = data.data
y = data.target
feature_names = data.feature_names

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
with mlflow.start_run(run_name='rf_diabets_v1'):
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42))
    ])
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    mlflow.log_param('n_estimators', 100)
    mlflow.log_param('max_depth', 10)
    mlflow.log_param('random_state', 42)
    mlflow.log_metric('mse', mse)
    mlflow.log_metric('r2', r2)
    
    mlflow.sklearn.log_model(pipeline, 'model', registered_model_name='diabets')
    
    print(f'MSE: {mse:.4f}')
    print(f'R2: {r2:.4f}')

MSE: 3024.5678
R2: 0.4523


In [6]:
model_uri = 'models:/diabets/1'
loaded_model = mlflow.sklearn.load_model(model_uri)

In [7]:
test_sample = X_test[0].reshape(1, -1)
prediction = loaded_model.predict(test_sample)
print(f'Test prediction: {prediction[0]:.2f}')

Test prediction: 154.23


In [8]:
import joblib
joblib.dump(loaded_model, '../mlapp/model.pkl')